## Diploma Thesis: Multi-Block Replacement

The objective of this notebook is to extend work on introduction notebook. Here we will focus on create an MVP methodology on how to replace multiple blocks inside an input transformer model.  
The key difference is that in the case of multi-block replacements and error propagation occurs across the model (Grafting paper). To ensure efficiency we utilize again the calibration data  
for local scope working as well as knowledge distillation for fine-tune retraining.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint
from typing import Literal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
project_pwd = Path().cwd().parents[1]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

In [3]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

from scripts.utils import *
from configs.intro_config import *

Model config - extended to multi-block workflow

In [4]:
mcfg = MCFG()
torch.manual_seed(mcfg.seed)

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

### Workflow

1) Load model and tokenizer
2) make data loader
3) collect blocks i/o pairs
4) evaluate block importance
5) select replacement strategy
6) apply replacement strategy on blocks
7) fine-tuning based on replacement strategy
8) eval

In [6]:
REPLACEMENT_OPERATOR = {
    'linear' : None
}

REPLACEMENT_STRATEGY = {
    'one_shot' : None,
    'sequential' : None
}